# 第8回 遺伝子発現データで、教師あり解析を一周する


---


## 前回からの接続


第2回から第7回までで、教師あり学習（答えの列がある解析）の道具がそろいました。前処理をする、式を作る、交差検証で評価する、正則化（regularization）で係数を落ち着かせる、前処理とモデルを1本のパイプライン（pipeline）にする、結果を解釈する。

今日は、新しいデータで、この道具を最初から最後まで一周します。データは、遺伝子の発現量です。手がかりの列が、行の数よりずっと多いデータです。これまでの道具を、そのまま使えるでしょうか。


---


## 今日の分析目標


あなたは眼科の研究者です。ほかの遺伝子の発現量から、TRIM32という遺伝子の発現量を当てられるか。当てられたとして、一緒に動く遺伝子を特定できるかに答える。

ラット120匹の目の組織で測った、31,000本ほどの遺伝子の発現量を使います。手がかりが匹数よりずっと多いときに、当たりを正しく評価する方法と、その落とし穴を確かめます。

もう1つのねらいは、自分のコードを、関数に分けた1本のスクリプトにまとめられるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、目の病気を研究する眼科の研究者です。TRIM32は、網膜の病気（バルデー・ビードル症候群）の原因遺伝子の1つとして報告された遺伝子です（Chiangほか, 2006）。

遺伝子は、ばらばらに働くのではなく、いくつもが一緒に強まったり弱まったりします。TRIM32と一緒に動く遺伝子が分かれば、TRIM32がどんな働きの仲間なのかを考える手がかりになります。


### 手元のデータ

米国アイオワ大学のグループが、交配したラット120匹の目の組織で、遺伝子の発現量を測ったデータです（Scheetzほか, 2006。NCBI GEOのGSE5680）。

1行が1匹です。列は、プローブ（遺伝子の断片を測るための印）ごとの発現量で、31,098本あります。値は、発現量を2を底とする対数（log2）にしたもので、1大きいと発現量が2倍です。TRIM32のプローブだけを、先頭の列`TRIM32`に出してあります。

プローブと遺伝子の名前の対応は、別の表（`rat_eye_genes.csv`）にあります。遺伝子の名前が分かっているプローブは、7割ほどです。


### 最後に出す答え

ほかの遺伝子からTRIM32を当てる式と、その当たり（平均でどれだけ外れるか）です。そして、その式からTRIM32と一緒に動く遺伝子について、何が言えて、何が言えないかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 手がかりの列が行より多いデータ（p > n）で、普通の線形回帰（linear regression）の訓練の当たりが当てにならない理由を説明できる
- 答えを見ずに、使わない列を決められる（対照のプローブ、発現の低いプローブ）
- 正則化や列の絞り込みをパイプラインの中に入れ、交差検証で当たりを正しく評価できる
- 答えを使って列を先に絞ってから交差検証すると、当たりがよく見えすぎることを、答えを並べ替えたデータで確かめられる
- 階層的クラスタリングで、遺伝子を一緒に動くまとまりに分け、樹形図を読める
- 遺伝子のまとまりを1本の点数にする部品をクラスで書き、パイプラインに入れられる
- 今日のコードを関数に分けた1本のスクリプトにまとめ、引数を変えて候補を比べ直せる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

今日は、列を絞る道具`SelectKBest`と`f_regression`、階層的クラスタリングの道具`linkage`・`fcluster`・`dendrogram`（scipyというライブラリにあります）、自分の部品をscikit-learnの部品にする`BaseEstimator`が加わります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.base import BaseEstimator
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：2つの表を読み込む

- 役割：発現量の表と、プローブと遺伝子の名前の表を読み込み、大きさを見る
- 入力：データの置き場所`DATA_DIR`と、2つのファイル名
- 出力：2つの表の大きさ

発現量の表は、大きいので圧縮してあります（`.csv.gz`）。`read_csv`は、圧縮したファイルもそのまま読めます。`index_col=0`は、1列目（ラットの番号、プローブの番号）を行の名前にする指定です。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/rat_eye_full.csv.gz', index_col=0)
genes = pd.read_csv(f'{DATA_DIR}/rat_eye_genes.csv', index_col=0)
print(df.shape)
print(genes.shape)


発現量の表は120行 × 31,099列で、120匹のラットに、TRIM32の列と31,098本のプローブの列が並びます。遺伝子の名前の表は、プローブ1本が1行です。


### お手本：答えと手がかりに分け、訓練とテストに分ける

- 役割：TRIM32を答え`y`、残りを手がかり`X`にし、訓練75%・テスト25%に分ける
- 入力：表`df`
- 出力：プローブの本数、訓練とテストの匹数

120匹しかいないので、テストを30匹にします（`test_size=0.25`）。テストは、最後に1回だけ使います。


In [ ]:
y = df['TRIM32']
X = df.drop(columns='TRIM32')
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0)
print(X.shape[1], '本のプローブ')
print(len(X_tr), len(X_te))


手がかりは31,098本のプローブです。訓練90匹、テスト30匹です。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は総合演習なので、3つのフェーズを全部通ります。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の8つの小さな問いに答えていきます。

1. どんなデータか？（1節）
2. どの列を使わないか？（2節）
3. 普通の式では？（3節）
4. ラッソなら？（4節）
5. 先に絞ると？（5節）
6. 遺伝子をまとまりに分けると？（6節）
7. どの候補を選ぶか？（7節）
8. 何が言えるか？（8節）


---


## 1. どんなデータか？


表が読めました。式を作る前に、中身をのぞきます。


### お手本：表の左上をのぞく

- 役割：訓練の先頭3匹・先頭4本のプローブを見る
- 入力：訓練の手がかり`X_tr`
- 出力：表の左上

列の名前の`1367452_at`などが、プローブの番号です。


In [ ]:
X_tr.iloc[:3, :4]


値は5〜11くらいの数です。log2なので、1違うと発現量は2倍違います。


### 変えてみる：答えのTRIM32を見る

- 役割：訓練のTRIM32の平均・標準偏差（standard deviation）・最小・最大を出す
- 入力：訓練の答え`y_tr`
- 出力：要約の数


In [ ]:
print(y_tr.describe().round(3))


TRIM32の発現量は、平均8.388、標準偏差0.152です。ラットごとの違いは小さく、log2で0.1や0.2の違いを当てることになります。外れの物差しは、これまでと同じ平均絶対誤差（MAE）です。


In [ ]:
plt.hist(y_tr, bins=20)
plt.xlabel('TRIM32 の発現量（log2）')
plt.ylabel('匹数')
plt.show()


ほとんどのラットは8.2〜8.6に集まり、左に、ほかより低い数匹がいます。


### 列が、行よりずっと多い

手がかりは31,098本、訓練は90匹です。手がかりの数を$p$、行の数を$n$と書くと、$p$が$n$の300倍以上あります。このようなデータを、$p > n$（「ピー大なりエヌ」）のデータと呼びます。遺伝子の発現量のデータは、たいていこの形です。

第5回で、30本の手がかりで完全な多重共線性（multicollinearity）に出会いました。今日は、それよりずっと極端な状況です。


In [ ]:
print(round(X_tr.shape[1] / X_tr.shape[0], 1))


$p / n$は、345.5です。


---


## 2. どの列を使わないか？


手がかりが3万本以上あります。全部を使う前に、第5回で考えた「使わない」の判断をします。ここで大事なのは、答え（TRIM32）を見ずに決めることです。


### お手本：対照のプローブを数える

プローブの中には、遺伝子を測るためではなく、測定がうまくいったかを確かめるための対照のプローブがあります。名前が`AFFX`で始まります。

- 役割：名前が`AFFX`で始まる列を数える
- 入力：手がかりの列の名前`X_tr.columns`
- 出力：対照のプローブの数

`X_tr.columns.str.startswith('AFFX')`は、列の名前1つずつについて、`AFFX`で始まればTrue、そうでなければFalseの並びを返します。`sum()`でTrueを数えるのは、第6回と同じです。


In [ ]:
affx = X_tr.columns.str.startswith('AFFX')
print(affx.sum(), '本が対照のプローブ')


57本が対照のプローブです。目の組織の遺伝子ではないので、使いません。


### 変えてみる：対照のプローブを外す

- 役割：対照のプローブ以外の列だけを残す
- 入力：訓練とテストの手がかり、印`affx`
- 出力：残った列の数

`X_tr.loc[:, 条件]`は、条件がTrueの列だけを残します（第1回の行の絞り方を、列に使う形です）。`~`は、TrueとFalseを入れ替えます。


In [ ]:
X_tr = X_tr.loc[:, ~affx]
X_te = X_te.loc[:, ~affx]
print(X_tr.shape[1], '本')


31,041本になりました。


### 発現の低いプローブ

第5回で、測った値が信頼できない列は使わないほうがよいと学びました。遺伝子発現のデータでは、発現の低いプローブがその例です。発現が低いと、測定の機械が拾う値は、雑音に近くなります。

- 役割：訓練の90匹で、プローブごとの発現量の平均を出し、その分布を見る
- 入力：訓練の手がかり`X_tr`
- 出力：平均の中央値と、平均のヒストグラム


In [ ]:
mean_tr = X_tr.mean()                  # プローブごとの平均（訓練）
print(round(mean_tr.median(), 2))
plt.hist(mean_tr, bins=50)
plt.axvline(mean_tr.median(), color='gray', ls='--')
plt.xlabel('プローブの発現量の平均（log2）')
plt.ylabel('本数')
plt.show()


プローブの平均は、低いほう（3〜5）にも山があります。発現の平均の中央値は5.87です。ここでは、平均が中央値以下の半分を、発現の低いプローブとして使わないことにします。中央値で切るのは、決め方の1つにすぎません。決めた基準は、報告に書き残します。


### TODO①：残る列を数える

`mean_tr > mean_tr.median()`で、平均が中央値より大きい列の印`keep`を作り、Trueの数（残る列の数）と、Falseの数（外す列の数）を表示してください。


In [ ]:
# TODO: 平均が中央値より大きい列の印 keep を作り、残す数・外す数を出す
# ヒント: keep = mean_tr > mean_tr.median()
#         keep.sum() と (~keep).sum()
...


残るのは15,520本、外すのは15,521本です。中央値で切ったので、ほぼ半分ずつになります。


### お手本：発現の低いプローブを外す

- 役割：印`keep`がTrueの列だけを、訓練とテストに残す
- 入力：訓練とテストの手がかり、平均`mean_tr`
- 出力：残った訓練の表の大きさ


In [ ]:
keep = mean_tr > mean_tr.median()
X_tr = X_tr.loc[:, keep]
X_te = X_te.loc[:, keep]
print(X_tr.shape)


訓練は90匹 × 15,520本になりました。対照のプローブと発現の低い半分を外しました。

ここまで、答えのTRIM32は一度も使っていません。平均は訓練の90匹だけで計算しました（第3回の「訓練だけで決める」）。

なお、試してみると、発現の低い半分を外しても、あとで作る式の当たりはほとんど変わりません。外す理由は、当たりのためではなく、計算を軽くし、雑音に近い列を「効いている遺伝子」として読んでしまわないためです。


---


## 3. 普通の式では？


使う列が15,520本に決まりました。まずは、これまでの普通の線形回帰で当ててみます。比べる相手として、全員を訓練の平均で当てたときのMAEも出します。


### お手本：普通の線形回帰で当てる

- 役割：15,520本で線形回帰の式を作り、訓練とテストのMAEを出す。平均で当てたときのテストのMAEも出す
- 入力：訓練とテストの手がかりと答え
- 出力：訓練のMAE、テストのMAE、平均で当てたときのテストのMAE


In [ ]:
def mae(y, pred):
    return np.abs(y - pred).mean()

ols = LinearRegression().fit(X_tr, y_tr)
print(round(mae(y_tr, ols.predict(X_tr)), 4))     # 訓練
print(round(mae(y_te, ols.predict(X_te)), 4))     # テスト
print(round(mae(y_te, y_tr.mean()), 4))           # 平均で当てると


訓練のMAEは0です。90匹の全員を、ぴったり当てました。ところが、テストのMAEは0.0787で、平均で当てたときの0.0913と比べて、少しよいだけです。


### なぜ訓練でぴったり当たるのか

式を作るのは、係数を決めることでした。係数は、15,520本の列それぞれに1つと、切片で、15,521個あります。一方、当てる相手は90匹です。

未知数が15,521個で、式が90本の連立方程式を考えます。未知数のほうがずっと多いので、90匹全員をぴったり当てる係数は、無数にあります。訓練でぴったり当たるのは、式がよいからではなく、自由度（degrees of freedom）が多すぎるからです。訓練の当たりは、まったく当てになりません。


In [ ]:
# @title 図：未知数が式より多い
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 3)
ax.add_patch(plt.Rectangle((0.5, 0.6), 7.5, 1.4, facecolor='#d8ecfb', edgecolor='#0066cc'))
ax.text(4.25, 1.3, '手がかり 15,520本（未知数 15,521個）', ha='center', fontsize=13)
ax.add_patch(plt.Rectangle((8.4, 0.6), 0.6, 1.4, facecolor='#fde2e4', edgecolor='#e63946'))
ax.text(8.7, 2.3, '90匹', ha='center', fontsize=13)
ax.text(4.25, 2.3, '横長の表：ぴったり当てる係数は無数にある', ha='center', fontsize=13)
plt.show()


### 交差検証で比べる準備

テストは最後に1回だけ使いたいので、ここからの候補の比較は、訓練の90匹の交差検証で行います（第4回）。比べる相手の、平均で当てたときの交差検証のMAEも出しておきます。

- 役割：訓練の90匹を5つの組に分け、各回で、その回の訓練の平均で当てたときのMAEを出す
- 入力：訓練の答え`y_tr`
- 出力：5回分のMAEと、その平均


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
base = []
for tr_i, te_i in kf.split(X_tr):
    base.append(mae(y_tr.iloc[te_i], y_tr.iloc[tr_i].mean()))
print(np.round(base, 4))
print(round(np.mean(base), 4))


平均で当てると、交差検証のMAEは0.0959です。これより小さくできれば、遺伝子の情報が役に立っている、と言えます。


---


## 4. ラッソなら？


普通の式は、訓練では当たっても、テストではあまり当たりませんでした。係数が自由に動きすぎるからです。第5回では、係数に罰をかけて落ち着かせました。罰が強ければ、ラッソは多くの係数をちょうど0にします。15,520本のうち、本当に使う遺伝子を、ラッソに選ばせてみます。


### おさらい：LassoCVとパイプライン

`LassoCV`は、罰の強さαを、交差検証で自分で選ぶラッソです（第5回の発展）。罰の前に標準化（standardization）するので、`StandardScaler`とつないで1本のパイプラインにします（第7回）。

```python
lasso = make_pipeline(StandardScaler(), LassoCV())
```


### お手本：ラッソのパイプラインを交差検証する

- 役割：標準化と`LassoCV`のパイプラインを作り、訓練の90匹の5つの組で交差検証して、MAEを出す
- 入力：訓練の手がかり`X_tr`と答え`y_tr`、分け方`kf`
- 出力：5回分のMAEと、その平均

`cross_val_score`の符号は、第5回で読んだとおり、マイナスで返ってくるので`-`を付けます。`max_iter=5000`は、収束（convergence）の警告を防ぐために、くり返しの回数を増やす指定です（第5回）。列が多いので、このセルは1分ほどかかります。


In [ ]:
lasso = make_pipeline(StandardScaler(),
                      LassoCV(cv=5, random_state=0, max_iter=5000))
scores = -cross_val_score(lasso, X_tr, y_tr, cv=kf,
                          scoring='neg_mean_absolute_error')
print(scores.round(4))
print(round(scores.mean(), 4))


交差検証のMAEは0.0756です。平均で当てた0.0959より、2割ほど小さくなりました。遺伝子の情報は、役に立っています。


### 交差検証が2重になっている

このセルでは、交差検証が2つ重なっています。

1. 外側：`cross_val_score`が、90匹を5つの組に分け、1つの組を評価用に取っておく
2. 内側：残りの72匹の中で、`LassoCV`が、さらに5つの組に分けてαを選ぶ

αを選ぶのは、答えを見て決める手順です。それを外側の評価用の組を見ずに行うので、評価用の組は、本当に「まだ見ていない匹」のままです。このように交差検証を重ねる形を、入れ子の交差検証（nested cross-validation）と呼びます。パイプラインに`LassoCV`を入れたので、入れ子は自動でできています。


In [ ]:
# @title 図：入れ子の交差検証
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 3.2)
for c in range(5):
    ax.add_patch(plt.Rectangle((0.5 + c * 1.6, 2.2), 1.5, 0.6,
                 facecolor='#fde2e4' if c == 4 else '#d8ecfb', edgecolor='white'))
ax.text(8.1, 2.95, '外側：評価用の組', ha='center', fontsize=12, color='#e63946')
for c in range(5):
    ax.add_patch(plt.Rectangle((0.5 + c * 1.24, 0.8), 1.18, 0.5,
                 facecolor='#fff1c1' if c == 4 else '#e3f4ef', edgecolor='white'))
ax.text(3.6, 0.35, '内側：残りの72匹で、さらに5つに分けて α を選ぶ', ha='center', fontsize=12)
ax.annotate('', xy=(3.6, 1.4), xytext=(3.6, 2.1), arrowprops=dict(arrowstyle='->', lw=2, color='#555'))
plt.show()


### お手本：訓練の全員で作り直し、選ばれた遺伝子の数を見る

- 役割：訓練の90匹全員でパイプラインを作り直し、選ばれたαと、係数が0でない列の数を出す
- 入力：パイプライン`lasso`、訓練の手がかりと答え
- 出力：αと、0でない係数の数

`lasso[-1]`は、パイプラインの最後の部品（`LassoCV`）です（第7回）。


In [ ]:
lasso.fit(X_tr, y_tr)
print(round(lasso[-1].alpha_, 4))
print((lasso[-1].coef_ != 0).sum(), '本が0でない')


αは0.0012で、係数が0でない列は87本です。15,520本のうち、0.6%ほどしか使っていません。


### TODO②：選ばれた遺伝子の名前を見る

ラッソの係数を、列の名前を付けた`pd.Series`にし、0でない係数のうち、絶対値の大きい5本のプローブと、その遺伝子の名前を表示してください。遺伝子の名前は、表`genes`の`symbol`の列にあります。


In [ ]:
# TODO: 係数の絶対値が大きい5本の、係数と遺伝子の名前を表示する
# ヒント: coef = pd.Series(lasso[-1].coef_, index=X_tr.columns)
#         top5 = coef.abs().sort_values(ascending=False).index[:5]
#         genes.loc[top5, 'symbol'] で名前
...


5本のうち、名前のないプローブ（`NaN`）もあります。遺伝子の名前が分かっていても、受講者のみなさんにはなじみのない名前ばかりでしょう。研究の場では、ここから文献を調べ、その遺伝子がどんな働きを持つかを確かめていきます。今日は、選ばれた遺伝子をどこまで信じてよいかを、6節と8節で考えます。


---


## 5. 先に絞ると？


ラッソは、15,520本から87本を選びました。もっと単純な方法も考えられます。TRIM32と相関の強い遺伝子を、先に数本だけ選び、その数本で普通の線形回帰をする方法です。


### お手本：相関の強い10本を選んで当てる

- 役割：訓練の90匹で、各プローブとTRIM32の相関係数を出し、絶対値の大きい10本を選ぶ。その10本の線形回帰を交差検証する
- 入力：訓練の手がかりと答え
- 出力：選んだ10本の相関係数と、交差検証のMAE

`X_tr.corrwith(y_tr)`は、`X_tr`の列1本ずつについて、`y_tr`との相関係数を出します。


In [ ]:
r = X_tr.corrwith(y_tr)                # 各プローブと TRIM32 の相関
top10 = r.abs().sort_values(ascending=False).index[:10]
print(r[top10].round(3).values)
pipe10 = make_pipeline(StandardScaler(), LinearRegression())
scores = -cross_val_score(pipe10, X_tr[top10], y_tr, cv=kf,
                          scoring='neg_mean_absolute_error')
print(round(scores.mean(), 4))


選んだ10本は、どれも相関係数の絶対値が0.79以上です。交差検証のMAEは0.057で、ラッソの0.0756よりずっと小さく見えます。


### 問い：0.057は、本当の当たりか？

10本を選んで普通の線形回帰をしただけで、ラッソよりずっと当たりました。この0.057を、まだ見ていないラットでの当たりとして報告してよいでしょうか。考えを下に書いてください。


（ここに考えを書く）


### 確かめ方：答えを並べ替える

確かめるために、TRIM32の値を、ラットの間でばらばらに並べ替えます。並べ替えると、どの遺伝子とも本当の関係はなくなります。関係のないデータで当たって見えたら、その手順はおかしい、と分かります。

- 役割：答えを並べ替えた`y_shuf`を作り、お手本と同じ手順（相関の強い10本を選んで交差検証）を行う。平均で当てたときのMAEも出す
- 入力：訓練の手がかりと答え
- 出力：並べ替えた答えでの、10本の交差検証のMAEと、平均で当てたときのMAE

`np.random.default_rng(0).permutation(...)`は、並びをばらばらに並べ替えます（`0`は並べ替え方の番号）。


In [ ]:
rng = np.random.default_rng(0)
y_shuf = pd.Series(rng.permutation(y_tr.values), index=y_tr.index)
r_s = X_tr.corrwith(y_shuf)
top_s = r_s.abs().sort_values(ascending=False).index[:10]
scores = -cross_val_score(pipe10, X_tr[top_s], y_shuf, cv=kf,
                          scoring='neg_mean_absolute_error')
base_s = []
for tr_i, te_i in kf.split(X_tr):
    base_s.append(mae(y_shuf.iloc[te_i], y_shuf.iloc[tr_i].mean()))
print(round(scores.mean(), 4), round(np.mean(base_s), 4))


関係のないデータなのに、10本の式の交差検証のMAEは0.0863で、平均で当てた0.0954より小さくなりました。当たって見えるのです。


### なぜ当たって見えるのか

10本を選ぶとき、90匹全員の答えを使いました。その90匹には、交差検証の各回で評価用に取っておく18匹も入っています。つまり、評価用の匹の答えを見て、その匹にたまたま合う遺伝子を選んでいたのです。

15,520本もあれば、本当は関係がなくても、たまたま答えと相関の強い遺伝子が必ず見つかります。それを選んで評価すれば、当たって見えます。第2回のリーク（leakage）、つまり答えの情報が手がかりに漏れ込むことと同じことが、選び方の手順を通して起きています。


### お手本：選ぶ手順を、パイプラインの中に入れる

正しくは、10本を選ぶ手順も、交差検証の各回の中で、その回の訓練の匹だけを使って行います。そのための部品が`SelectKBest`です。

- 役割：標準化 → 相関の強い10本を選ぶ → 線形回帰、のパイプラインを作り、交差検証する
- 入力：訓練の手がかり（15,520本すべて）と答え
- 出力：交差検証のMAE

`SelectKBest(f_regression, k=10)`は、`fit`のときに、渡された匹で、答えとの関係の強さ（`f_regression`）を列ごとに測り、強い10本を覚えます。`transform`では、その10本だけを残します。`f_regression`の強さは、相関係数の絶対値と同じ順に並ぶので、相関の強い10本を選ぶことになります。


In [ ]:
sel10 = make_pipeline(StandardScaler(),
                      SelectKBest(f_regression, k=10),
                      LinearRegression())
scores = -cross_val_score(sel10, X_tr, y_tr, cv=kf,
                          scoring='neg_mean_absolute_error')
print(scores.round(4))
print(round(scores.mean(), 4))


正しく評価すると、交差検証のMAEは0.0685です。先に選んだときの0.057より大きくなりましたが、ラッソの0.0756よりは小さく、平均の0.0959よりずっと小さい値です。相関の強い遺伝子で当てる方法は、正しく評価しても、役に立っています。


### TODO③：並べ替えた答えで、正しく評価する

お手本の`sel10`を、並べ替えた答え`y_shuf`で交差検証し、MAEの平均を表示してください。平均で当てたときの0.0954と比べましょう。


In [ ]:
# TODO: sel10 を y_shuf で交差検証し、MAE の平均を表示する
# ヒント: お手本の cross_val_score の y_tr を y_shuf に変える
...


MAEは0.1228で、平均で当てた0.0954より大きくなります。関係のないデータでは、正しく評価すれば、当たらないと分かります。

答えを使う手順（列を選ぶ、αを選ぶ、標準化の平均を決める）は、すべてパイプラインに入れて、交差検証の中で行います。


---


## 6. 遺伝子をまとまりに分けると？


ここまでの2つの方法は、どちらも答えのTRIM32を使って遺伝子を選びました。今度は逆に、答えを見ずに、遺伝子どうしの動き方だけから、一緒に動く遺伝子のまとまりを探します。まとまりが見つかったら、あとで答えと照らします。

遺伝子をまとまりに分けるのに、階層的クラスタリング（hierarchical clustering）という方法を使います。初めての方法なので、5本の小さな例から始めます。


### お手本：5本の遺伝子の相関を見る

- 役割：5本のプローブを選び、訓練の90匹での相関係数の表を作る
- 入力：5本のプローブの番号`ex`、訓練の手がかり、遺伝子の名前の表
- 出力：5本の遺伝子の名前と、相関係数の表

`.rename(index=..., columns=...)`で、行と列の名前を、プローブの番号から遺伝子の名前に付け替えます。


In [ ]:
ex = ['1383110_at', '1394033_at', '1376559_at', '1371242_at',
      '1379101_at']
names = genes.loc[ex, 'symbol']
print(names.values)
corr5 = X_tr[ex].corr().rename(index=names, columns=names)
print(corr5.round(2))


Klhl24とRbbp9は相関0.83、RpiaとGrm8は0.90で、それぞれ一緒に動きます。Klhl24とRpiaは−0.70で、逆向きに動きます。Dhx36は、ほかのどれとも相関が0に近く、別の動き方をしています。


### 遺伝子どうしの距離

一緒に動く遺伝子ほど「近い」と考えます。相関係数$r$から、距離$d$を次のように決めます。

$$
d = 1 - r
$$

一緒に動く（$r = 1$）なら距離0、関係がない（$r = 0$）なら距離1、逆向きに動く（$r = -1$）なら距離2です。

- 役割：5本の遺伝子の距離の表を作る
- 入力：相関係数の表`corr5`
- 出力：距離の表


In [ ]:
dist5 = 1 - corr5
print(dist5.round(2))


いちばん近いのは、RpiaとGrm8（0.1）で、次がKlhl24とRbbp9（0.17）です。


### 近いものから、順にまとめる

階層的クラスタリングは、次の手順をくり返します。

1. はじめは、5本の遺伝子が、それぞれ1本ずつのまとまり
2. いちばん近い2つのまとまりを、1つにまとめる
3. まとまりが1つになるまで、2をくり返す

5本の例では、まずRpiaとGrm8（距離0.1）、次にKlhl24とRbbp9（距離0.17）がまとまります。


### まとまりどうしの距離：群平均法

2本がまとまったあとは、まとまりとまとまりの距離が必要です。ここでは、2つのまとまりから1本ずつ選んだ組の距離を、すべて平均したものにします。この決め方を、群平均法（average linkage）と呼びます。

まとまり$A$と$B$の遺伝子の数を$|A|$、$|B|$とすると、

$$
D(A, B) = \frac{1}{|A|\,|B|} \sum_{a \in A} \sum_{b \in B} d(a, b)
$$

です。Dhx36と、{Klhl24, Rbbp9}の距離は、Dhx36とKlhl24の距離0.98と、Dhx36とRbbp9の距離1.0の平均で、0.99です。Dhx36と{Rpia, Grm8}の距離は、1.01と1.06の平均で、0.99より大きくなります。小さいほう、{Klhl24, Rbbp9}とDhx36が、3番目にまとまります。


### お手本：linkageで、まとめる手順を計算する

- 役割：5本の遺伝子を、群平均法と距離$1 - r$でまとめる手順を計算する
- 入力：訓練の5本の列`X_tr[ex]`
- 出力：まとめた手順の表（1行が1回のまとめ）

`linkage`は、行をまとめていきます。今は遺伝子（列）をまとめたいので、`.T`で表の行と列を入れ替えて（転置して）渡します。`'average'`は群平均法、`metric='correlation'`は距離$1 - r$の指定です。


In [ ]:
Z5 = linkage(X_tr[ex].T, 'average', metric='correlation')
print(Z5.round(3))


1行が1回のまとめで、左から「まとめた2つの番号、そのときの距離、まとまりの遺伝子の数」です。番号の0〜4は、5本の遺伝子（`ex`の順）、5以上は、それまでにできたまとまり（5が1回目、6が2回目…）です。

1回目は2番（Rpia）と3番（Grm8）を距離0.105で、2回目は0番（Klhl24）と1番（Rbbp9）を0.174で、3回目は4番（Dhx36）と6番（2回目のまとまり）を0.991でまとめています。手で計算したとおりです。


### お手本：樹形図に描く

- 役割：まとめた手順を、樹形図に描く
- 入力：手順の表`Z5`と、遺伝子の名前
- 出力：樹形図

まとめた手順を描いた図を、樹形図（dendrogram）と呼びます。縦の高さが、まとめたときの距離です。


In [ ]:
dendrogram(Z5, labels=names.values)
plt.ylabel('まとめたときの距離')
plt.show()


低い位置でつながっている遺伝子ほど、一緒に動きます。RpiaとGrm8、Klhl24とRbbp9が、低い位置でまとまっています。


### お手本：いくつのまとまりに切るか

樹形図を、ある高さで横に切ると、その下でつながっている遺伝子が、1つのまとまりになります。`fcluster`は、まとまりの数を決めて切ります。

- 役割：5本の遺伝子を、2つのまとまりに切る
- 入力：手順の表`Z5`
- 出力：遺伝子ごとの、まとまりの番号

`'maxclust'`は、「まとまりの数が2以下になるように切る」という指定です。


In [ ]:
print(fcluster(Z5, 2, 'maxclust'))


Klhl24・Rbbp9・Dhx36が1つ、Rpia・Grm8がもう1つのまとまりになりました。これが、階層的クラスタリングの全体です。


### お手本：2,000本を、10のまとまりに分ける

5本の例と同じことを、2,000本で行います。2,000本は、答えを見ずに、訓練の90匹での分散（ばらつき）の大きい順に選びます。ばらつきの小さい遺伝子は、どのラットでもほぼ同じなので、一緒に動いているかを判断しにくいからです。

- 役割：分散の大きい2,000本を、群平均法で10のまとまりに分け、まとまりごとの遺伝子の数を出す
- 入力：訓練の手がかり
- 出力：まとまりごとの遺伝子の数

`pd.Series(…, index=var_top)`で、まとまりの番号に、プローブの番号を付けます。


In [ ]:
var_top = X_tr.var().sort_values(ascending=False).index[:2000]
Z = linkage(X_tr[var_top].T, 'average', metric='correlation')
groups = pd.Series(fcluster(Z, 10, 'maxclust'), index=var_top)
print(groups.value_counts().sort_index())


10のまとまりの大きさは、1,155本・527本・143本・115本のような大きいものから、3本・7本の小さいものまで、さまざまです。


In [ ]:
# @title 図：2,000本の相関（樹形図の順に並べる）
order = dendrogram(Z, no_plot=True)['leaves']
cm = np.corrcoef(X_tr[var_top].T.values)[np.ix_(order, order)]
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='相関係数')
plt.title('2,000本の遺伝子の相関（樹形図の順）')
plt.xticks([]); plt.yticks([])
plt.show()


2,000本どうしの相関を、樹形図の順に並べて色で描きました。赤は一緒に動く、青は逆向きに動く組です。対角線に沿って、赤い四角がいくつも見えます。それぞれの四角が、一緒に動く遺伝子のまとまりです。


### お手本：まとまりを1本の点数にし、答えと照らす

まとまりの遺伝子は一緒に動くので、まとめて1本の点数にします。遺伝子を標準化してから、まとまりの中で平均します。

- 役割：2,000本を標準化し、まとまりごとに平均した点数を作り、TRIM32との相関係数を出す
- 入力：訓練の手がかり、まとまりの番号`groups`、答え
- 出力：まとまりごとの、点数とTRIM32の相関係数

ここで、初めて答えのTRIM32を使います。


In [ ]:
Zs = (X_tr[var_top] - X_tr[var_top].mean()) / X_tr[var_top].std()
score = pd.DataFrame()
for g in range(1, 11):
    score[g] = Zs.loc[:, groups == g].mean(axis=1)   # g の平均
print(score.corrwith(y_tr).round(3))


7番のまとまり（527本）の点数は、TRIM32と相関0.746で、一緒に強く動きます。1番のまとまり（19本）は−0.664で、逆向きに動きます。ほかのまとまりは、0.3以下です。答えを見ずに分けたまとまりの中に、TRIM32と一緒に動くまとまりが見つかりました。


### 平均すると、相関が強くなる

7番のまとまりの遺伝子を1本ずつ見ると、TRIM32との相関は、それほど強くありません。

- 役割：7番のまとまりの遺伝子1本ずつの、TRIM32との相関係数の中央値を出す
- 入力：訓練の手がかり、`groups`、答え
- 出力：相関係数の中央値


In [ ]:
r7 = X_tr[var_top].corrwith(y_tr)[groups == 7]
print(round(r7.median(), 3))


1本ずつでは、相関の中央値は0.469です。それが、527本を平均すると0.746になりました。1本ずつの測定には雑音がありますが、一緒に動く遺伝子を平均すると、雑音が打ち消し合い、共通の動きが残るのです。


### お手本：まとまりの点数を作る部品

まとまりを作る手順も、パイプラインに入れたい部品です。今回は答えを使いませんが、分散や平均を訓練で決めるので、交差検証の各回の中で行うのが正しい手順です。第7回の`Standardizer`と同じ形で、クラスにします。

- 役割：`fit`で、分散の大きい遺伝子を選び、まとまりに分け、平均と標準偏差を覚える。`transform`で、覚えた値で標準化し、まとまりごとの点数の表を返す
- 入力：なし（部品の設計図を書くだけ）
- 出力：なし

新しいのは2つです。1つ目の`__init__`は、部品を作るときに引数（遺伝子の数`n_genes`、まとまりの数`k`）を受け取って覚えるメソッドです。2つ目の`(BaseEstimator)`は、scikit-learnの部品の性質を受け継ぐ指定で、これを付けると`cross_val_score`に渡せます。


In [ ]:
class GroupScore(BaseEstimator):
    def __init__(self, n_genes=2000, k=10):
        self.n_genes = n_genes
        self.k = k

    def fit(self, X, y=None):
        v = X.var().sort_values(ascending=False)
        top = v.index[:self.n_genes]
        self.mean = X[top].mean()
        self.std = X[top].std()
        Z = linkage(X[top].T, 'average', metric='correlation')
        self.groups = pd.Series(fcluster(Z, self.k, 'maxclust'),
                                index=top)
        return self

    def transform(self, X):
        Zs = (X[self.groups.index] - self.mean) / self.std
        out = pd.DataFrame()
        for g in sorted(self.groups.unique()):
            out[g] = Zs.loc[:, self.groups == g].mean(axis=1)
        return out


ブロックで読みます。

- `__init__`：引数を`self`に覚えるだけ
- `fit`：今日のセルを並べたもの。分散の大きい遺伝子を選ぶ → 平均と標準偏差を覚える → `linkage`と`fcluster`でまとまりに分ける
- `transform`：覚えた平均と標準偏差で標準化し、まとまりごとに平均した点数の表を返す

`fit(self, X, y=None)`の`y=None`は、「答えを受け取ってもよいが、使わない」という形です。パイプラインは、どの部品にも答えを渡そうとするからです。


### お手本：まとまりの点数で当てるパイプライン

- 役割：まとまりの点数 → 線形回帰、のパイプラインを作り、交差検証する
- 入力：訓練の手がかり（15,520本）と答え
- 出力：5回分のMAEと、その平均


In [ ]:
grp = make_pipeline(GroupScore(n_genes=2000, k=10),
                    LinearRegression())
scores = -cross_val_score(grp, X_tr, y_tr, cv=kf,
                          scoring='neg_mean_absolute_error')
print(scores.round(4))
print(round(scores.mean(), 4))


交差検証のMAEは0.0738で、ラッソの0.0756と同じくらいです。10本の点数だけで、ラッソと同じくらい当たりました。


### TODO④：まとまりの数を5にする

お手本のパイプラインで、`k=10`を`k=5`にして交差検証し、MAEの平均を表示してください。


In [ ]:
# TODO: k=5 の GroupScore で交差検証し、MAE の平均を表示する
# ヒント: grp5 = make_pipeline(GroupScore(n_genes=2000, k=5),
#                              LinearRegression())
...


MAEは0.0725です。10のときの0.0738と、ほとんど変わりません。


---


## 7. どの候補を選ぶか？


当てる方法の候補が3つそろいました。ラッソ（0.0756）、相関の強い10本（0.0685）、まとまりの点数（0.0738）です。最後の1つを選んでテストで確かめる前に、今日のコードを、関数に分けた1本のスクリプトにまとめます。


### なぜ、まとめるのか

今日のノートは、上から順にセルを実行すると動きます。ただ、2節で`X_tr`を書き換えたように、途中のセルを飛ばしたり、2回実行したりすると、結果が変わってしまいます。

手順を関数に分けて1本のスクリプトにすると、上から1回実行するだけで、いつも同じ結果になります。候補を比べ直すときも、関数の引数を変えるだけで済みます。論文のコードも、たいていこの形で書かれています。


### お手本：読み込みと、列の選び方を関数に

- 役割：データを読み込んで訓練とテストに分ける関数`load`と、使わない列を外す関数`choose_columns`を作る
- 入力：なし（関数を定義するだけ）
- 出力：なし

中身は、0節と2節のセルを並べたものです。


In [ ]:
def load():
    df = pd.read_csv(f'{DATA_DIR}/rat_eye_full.csv.gz', index_col=0)
    y = df['TRIM32']
    X = df.drop(columns='TRIM32')
    return train_test_split(X, y, test_size=0.25, random_state=0)


def choose_columns(X_tr, X_te):
    affx = X_tr.columns.str.startswith('AFFX')
    X_tr, X_te = X_tr.loc[:, ~affx], X_te.loc[:, ~affx]
    mean_tr = X_tr.mean()
    keep = mean_tr > mean_tr.median()
    return X_tr.loc[:, keep], X_te.loc[:, keep]


### お手本：パイプラインを作る関数と、評価する関数

- 役割：候補の名前からパイプラインを作る関数`build_pipeline`と、交差検証のMAEの平均を返す関数`evaluate`を作る
- 入力：なし（関数を定義するだけ）
- 出力：なし

`build_pipeline`は、`name`で候補を選び、`k`で選ぶ本数（またはまとまりの数）を変えられるようにします。`k=10`は、何も書かなければ10という指定（既定値）です。


In [ ]:
def build_pipeline(name, k=10):
    if name == 'lasso':
        lasso_cv = LassoCV(cv=5, random_state=0, max_iter=5000)
        return make_pipeline(StandardScaler(), lasso_cv)
    if name == 'top':
        return make_pipeline(StandardScaler(),
                             SelectKBest(f_regression, k=k),
                             LinearRegression())
    if name == 'group':
        return make_pipeline(GroupScore(n_genes=2000, k=k),
                             LinearRegression())


def evaluate(pipe, X, y):
    scores = -cross_val_score(pipe, X, y, cv=kf,
                              scoring='neg_mean_absolute_error')
    return round(scores.mean(), 4)


### お手本：1本のスクリプトとして実行する

- 役割：4つの関数を順に呼び、3つの候補の交差検証のMAEを並べる
- 入力：なし
- 出力：候補ごとのMAE

このセルだけで、読み込みから比較までが動きます。ラッソがあるので、1分ほどかかります。


In [ ]:
X_tr, X_te, y_tr, y_te = load()
X_tr, X_te = choose_columns(X_tr, X_te)
for name in ['lasso', 'top', 'group']:
    print(name, evaluate(build_pipeline(name), X_tr, y_tr))


ラッソ0.0756、相関の強い10本0.0685、まとまりの点数0.0738で、これまでのセルと同じ数になりました。いちばん小さいのは、相関の強い10本です。

なお、選ぶ候補は、テストではなく交差検証のMAEで決めます。テストで選ぶと、テストが最後の確かめにならないからです（第5回のαと同じ考え方です）。


### TODO⑤：選んだ候補を、テストで1回だけ確かめる

`build_pipeline('top')`を訓練の90匹全員で`fit`し、テストの30匹のMAEを出してください。平均で当てたときのテストのMAE（3節の0.0913）と比べ、何割小さくなったかも表示してください。


In [ ]:
# TODO: 'top' のパイプラインを訓練で fit し、テストの MAE を出す
# ヒント: best = build_pipeline('top').fit(X_tr, y_tr)
#         mae(y_te, best.predict(X_te)) と mae(y_te, y_tr.mean())
...


テストのMAEは0.0603です。平均で当てたときの0.0913より、34%小さくなりました。


---


## 8. 何が言えるか？


当てる式が決まりました。最後に、問いのもう半分、「一緒に動く遺伝子を特定できるか」を考えます。


### お手本：選ばれる10本は、分け方で入れ替わるか

- 役割：交差検証の5回それぞれで、その回の訓練の72匹から相関の強い10本を選び、5回すべてで選ばれた本数を数える
- 入力：訓練の手がかりと答え、分け方`kf`
- 出力：回ごとの10本の、1回目との重なりと、5回すべてで選ばれた本数

`set`は、重なりのない集まりです。`&`で、2つの集まりの両方に入っているものを取り出します。


In [ ]:
chosen = []
for tr_i, te_i in kf.split(X_tr):
    r_i = X_tr.iloc[tr_i].corrwith(y_tr.iloc[tr_i])
    top_i = r_i.abs().sort_values(ascending=False).index[:10]
    chosen.append(set(top_i))
print([len(chosen[0] & c) for c in chosen])
print(len(set.intersection(*chosen)), '本が5回とも選ばれた')


1回目の10本と重なるのは、ほかの回では1〜4本だけです。5回とも選ばれた遺伝子は、0本でした。72匹の選び方が少し変わるだけで、「相関の強い10本」は入れ替わります。


### なぜ入れ替わるのか

6節で見たとおり、TRIM32と一緒に動く遺伝子は、何百本ものまとまりになっています。まとまりの中の遺伝子は互いに似ているので、どれを選んでも、当たりはあまり変わりません。分け方が変わると、まとまりの中の、たまたま相関が少し強かった別の遺伝子が選ばれます。

第4回で、似た列（多重共線性）があると係数が揺れることを、第5回で、ラッソが似た列のどちらかを0にすることを見ました。同じことが、何百本の規模で起きているのです。選ばれた10本は、「TRIM32を動かしている遺伝子」ではなく、「一緒に動くまとまりの、その時の代表」にすぎません。


### 答え

ほかの遺伝子の発現量から、TRIM32の発現量は当てられます。相関の強い10本を選ぶ手順をパイプラインに入れた式で、テストの30匹のMAEは0.0603で、平均で当てた0.0913より34%小さくなりました。

ただし、効いている遺伝子を特定することはできません。選ばれる遺伝子は分け方で入れ替わり、一緒に動く大きなまとまり（答えを見ずに分けた527本のまとまりは、TRIM32と相関0.746）の代表にすぎないからです。言えるのは、「TRIM32は、数百本の遺伝子と一緒に動いている」ことまでです。

そして、答えを使う手順を交差検証の外で行うと、関係のないデータでも当たって見えます（答えを並べ替えると、先に選んだ式のMAEは0.0863で、平均の0.0954より小さかった）。


まだ言えないこと

- 原因か：一緒に動くことは、どちらがどちらを動かしているかを教えない。確かめるには、遺伝子を働かなくした実験などが要る
- ほかの組織やほかの動物でも同じか：このデータは、ラットの目の組織だけ
- まとまりの生物学的な意味：まとまりの遺伝子がどんな働きを共有しているかは、文献やデータベースで調べる必要がある


---


## 目標に答えられたか


- 今日の目標は「ほかの遺伝子の発現量から、TRIM32の発現量を当てられるか。一緒に動く遺伝子を特定できるか」でした。答えを、MAEの数と、まとまりの言葉で説明できますか？
- 2節で、使わない列を決めるとき、答えを使わなかったのはなぜですか？
- 3節で、普通の線形回帰の訓練のMAEが0になったのはなぜですか？
- 4節の入れ子の交差検証で、内側と外側はそれぞれ何をしていましたか？
- 5節で、答えを並べ替えたのに当たって見えたのはなぜですか？ どう直しましたか？
- 6節の階層的クラスタリングで、距離と、まとまりどうしの距離は、どう決めましたか？ 樹形図の高さは何を表しますか？
- 6節で、まとまりを平均した点数の相関が、1本ずつより強くなったのはなぜですか？
- 7節で、関数に分けた1本のスクリプトにすると、何がよくなりましたか？
- 8節で、選ばれた遺伝子を「TRIM32を動かす遺伝子」と言えないのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：遺伝子が匹数より多くても、正則化や絞り込みをパイプラインの中に入れれば、当たりを正しく評価できる。答えを使う手順は、すべて交差検証の中に入れる。当たる式が作れても、効いている遺伝子を1本に特定できるとは限らない
- $p > n$では、普通の線形回帰は訓練でぴったり当たるが、訓練の当たりは当てにならない
- 使わない列（対照、発現の低いプローブ）は、答えを見ずに、訓練だけで決める
- ラッソや列の絞り込みも、パイプラインに入れて交差検証する（入れ子の交差検証）。外で選ぶと、関係のないデータでも当たって見える
- 階層的クラスタリング：距離$1 - r$で、近いまとまりから順にまとめる（群平均法）。樹形図を切って、まとまりを作る
- 一緒に動く遺伝子は、平均すると雑音が打ち消し合い、共通の動きが残る
- Python：手順を関数（`load`・`choose_columns`・`build_pipeline`・`evaluate`）に分けた1本のスクリプトにすると、いつも同じ結果になり、引数を変えるだけで比べ直せる


---


## 次回へ


今日は、2,000本の遺伝子を、一緒に動くまとまりに分け、まとまりごとの平均で、たくさんの列を10本ほどの点数に縮めました。平均という決め方は、いちばん単純な縮め方です。

次回の問いは、たくさんの数でできたデータを、できるだけ情報を失わずに、少ない数に縮めるには、どうすればよいか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

7節のスクリプトの関数を使い、相関の強い遺伝子を選ぶ本数`k`を5・20・50に変えて、`build_pipeline('top', k=k)`の交差検証のMAEを並べてください。`for`で`k`を回すと、関数を呼ぶだけで比べ直せます。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`for k in [5, 20, 50]:`の中で、`print(k, evaluate(build_pipeline('top', k=k), X_tr, y_tr))`とします。
本編の`k=10`は0.0685でした。

</details>


### 応用②（判断）

応用①の3つ（`k`=5・20・50）のうち、交差検証のMAEがいちばん小さいのはどれですか。そのMAEを、小数第4位まで答えてください（例: 0.1234）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

交差検証のMAEは、小さいほどよく当たります。テストのMAEで選ばないように注意しましょう。

</details>


### 応用③（解釈）

共同研究者（生物学の研究者）に向けて、今日の解析で分かったことを3行程度で報告してください。当たりの数（MAE）、選ばれた遺伝子をどこまで信じてよいか、次に何を調べるとよいか、を入れましょう。


（ここに3行程度で書く）


---


## 発展（任意）


### 同じ病気のほかの遺伝子は、一緒に動くか

TRIM32は、バルデー・ビードル症候群の原因遺伝子の1つです。この病気には、ほかにも原因遺伝子が知られています（Lztfl1・Bbs5・Ttc8・Arl6・Bbs2など）。これらの遺伝子が、TRIM32と一緒に動くかを見ます。

- 役割：5つの遺伝子のプローブを探し、訓練の90匹で、TRIM32との相関係数を出す。相関の強い10本や、7番のまとまりに入っているかも見る
- 入力：遺伝子の名前の表、訓練の手がかりと答え
- 出力：プローブごとの遺伝子の名前、相関係数、10本に入るか、7番のまとまりか

`isin`は、値が並びのどれかと同じならTrueを返します。


In [ ]:
bbs = ['Lztfl1', 'Bbs5', 'Ttc8', 'Arl6', 'Bbs2']
probes = genes.index[genes['symbol'].isin(bbs)]
probes = [p for p in probes if p in X_tr.columns]
r_bbs = X_tr[probes].corrwith(y_tr)
print(pd.DataFrame({'遺伝子': genes.loc[probes, 'symbol'],
                    '相関': r_bbs.round(3),
                    '10本に入る': [p in top10 for p in probes],
                    '7番': [groups.get(p) == 7 for p in probes]}))


どの遺伝子も、TRIM32と相関0.6以上（0.621〜0.73）で、一緒に動いています。それでも、相関の強い10本には1本も入りません。0.8を超える遺伝子がたくさんあるからです。分散の大きい2,000本にも入らないので、6節のまとまりにも入っていません。

「選ばれなかった＝関係がない」ではありません。同じ病気の遺伝子が一緒に動いていることは、TRIM32の働きを考える大事な手がかりですが、当たりのよい式を作る手順からは見えてきません。何を知りたいかで、解析の手順を選ぶ必要があります。

試すなら、`r.rank(ascending=False)[probes]`で、15,520本の中で何番目に相関が強いかを見てみましょう。


### まとまりの遺伝子の名前を見る

6節で、TRIM32と一緒に動いた7番のまとまり（527本）に、どんな遺伝子が入っているかを見ます。

- 役割：7番のまとまりのプローブのうち、TRIM32との相関の強い10本の遺伝子の名前を表示する
- 入力：まとまり`groups`、訓練の手がかりと答え、遺伝子の名前の表
- 出力：遺伝子の名前と相関係数


In [ ]:
in7 = groups.index[groups == 7]
r_in7 = X_tr[in7].corrwith(y_tr).sort_values(ascending=False)
print(pd.DataFrame({'遺伝子': genes.loc[r_in7.index[:10], 'symbol'],
                    '相関': r_in7[:10].round(3)}))


研究の場では、まとまりの遺伝子の一覧を、遺伝子の働きのデータベース（Gene Ontologyなど）と照らして、どんな働きが多く含まれるかを調べます。まとまりの意味は、データの外の知識と合わせて初めて読めます。
